This notebook evaluates how inventory recommendations change under different business conditions.

The optimization model from Notebook 08 is reused to test demand, capacity, holding cost, stockout cost, and safety stock assumptions.

In [1]:
# Configure notebook environment
from src.utils.project_setup import configure_notebook

configure_notebook()

# General libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Visualization
import plotly.express as px

# Reusable optimization function
from src.optimization.inventory_optimizer import run_inventory_optimization

Notebook configured successfully.


In [2]:
# Load optimization input created in Notebook 08
optimization_input = pd.read_csv(
    "../outputs/optimization_input.csv"
)

print(f"Optimization input shape: {optimization_input.shape}")

optimization_input.head()

Optimization input shape: (100, 13)


,item_id,store_id,state_id,forecast_demand,actual_demand,average_price,holding_cost,stockout_cost,recommended_inventory,expected_shortage,expected_excess,safety_stock,target_inventory
0,FOODS_2_025,TX_1,TX,10.519777,11,3.86,0.579,7.72,12.623733,0.0,0.0,2.103956,12.623733
1,FOODS_3_145,TX_3,TX,7.243583,6,2.98,0.447,5.96,8.692300,0.0,0.0,1.448717,8.692300
2,FOODS_3_261,CA_2,CA,42.509094,44,3.98,0.597,7.96,51.010914,0.0,0.0,8.501819,51.010914
3,FOODS_2_261,TX_3,TX,13.823271,15,9.88,1.482,19.76,16.587925,0.0,0.0,2.764654,16.587925
4,HOBBIES_1_118,WI_3,WI,4.516345,5,11.12,1.668,22.24,5.419614,0.0,0.0,0.903269,5.419614


In [3]:
# Validate required columns for the optimization function
required_columns = [
    "item_id",
    "store_id",
    "state_id",
    "forecast_demand",
    "actual_demand",
    "average_price",
]

missing_columns = [
    col for col in required_columns
    if col not in optimization_input.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Optimization input validation passed.")

Optimization input validation passed.


In [4]:
# Define scenarios
scenarios = [
    {
        "scenario_name": "Base Case",
        "demand_multiplier": 1.00,
        "capacity_rate": 0.95,
        "holding_cost_rate": 0.15,
        "stockout_cost_multiplier": 2.0,
        "safety_stock_rate": 0.20,
    },
    {
        "scenario_name": "Peak Demand",
        "demand_multiplier": 1.20,
        "capacity_rate": 0.95,
        "holding_cost_rate": 0.15,
        "stockout_cost_multiplier": 2.0,
        "safety_stock_rate": 0.20,
    },
    {
        "scenario_name": "Capacity Constraint",
        "demand_multiplier": 1.00,
        "capacity_rate": 0.80,
        "holding_cost_rate": 0.15,
        "stockout_cost_multiplier": 2.0,
        "safety_stock_rate": 0.20,
    },
    {
        "scenario_name": "High Stockout Penalty",
        "demand_multiplier": 1.00,
        "capacity_rate": 0.95,
        "holding_cost_rate": 0.15,
        "stockout_cost_multiplier": 4.0,
        "safety_stock_rate": 0.20,
    },
    {
        "scenario_name": "High Holding Cost",
        "demand_multiplier": 1.00,
        "capacity_rate": 0.95,
        "holding_cost_rate": 0.30,
        "stockout_cost_multiplier": 2.0,
        "safety_stock_rate": 0.20,
    },
    {
        "scenario_name": "Aggressive Service Strategy",
        "demand_multiplier": 1.00,
        "capacity_rate": 1.10,
        "holding_cost_rate": 0.15,
        "stockout_cost_multiplier": 4.0,
        "safety_stock_rate": 0.30,
    },
]

In [5]:
# Run optimization model for every scenario
all_detailed_results = []
all_summaries = []

for scenario in scenarios:

    detailed_results, summary = run_inventory_optimization(
        input_data=optimization_input,
        **scenario
    )

    all_detailed_results.append(detailed_results)
    all_summaries.append(summary)

# Combine scenario outputs
scenario_detailed_results = pd.concat(
    all_detailed_results,
    ignore_index=True
)

scenario_summary = pd.concat(
    all_summaries,
    ignore_index=True
)

scenario_summary

,scenario,solver_status,objective_value,total_forecast_demand,total_target_inventory,warehouse_capacity,total_recommended_inventory,total_actual_demand,total_actual_shortage,total_actual_excess,service_level,capacity_used,demand_multiplier,capacity_rate,holding_cost_rate,stockout_cost_multiplier,safety_stock_rate
0,Base Case,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,2.0,0.2
1,Peak Demand,Optimal,346.381198,3696.936912,4436.324294,4214.508080,4214.508079,3167,148.066172,1195.574251,0.953247,1.000000,1.2,0.95,0.15,2.0,0.2
2,Capacity Constraint,Optimal,1634.645594,3080.780760,3696.936912,2957.549530,2957.549529,3167,625.525758,416.075287,0.802486,1.000000,1.0,0.80,0.15,2.0,0.2
3,High Stockout Penalty,Optimal,577.301993,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,4.0,0.2
4,High Holding Cost,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.30,2.0,0.2
5,Aggressive Service Strategy,Optimal,0.000000,3080.780760,4005.014988,4405.516487,4005.014993,3167,13.059738,851.074732,0.995876,0.909091,1.0,1.10,0.15,4.0,0.3


In [6]:
# Save scenario analysis outputs
scenario_summary.to_csv(
    "../outputs/scenario_summary.csv",
    index=False
)

scenario_detailed_results.to_csv(
    "../outputs/scenario_detailed_results.csv",
    index=False
)

print("Scenario analysis outputs saved.")

Scenario analysis outputs saved.


In [7]:
# Review scenario-level results
scenario_summary

,scenario,solver_status,objective_value,total_forecast_demand,total_target_inventory,warehouse_capacity,total_recommended_inventory,total_actual_demand,total_actual_shortage,total_actual_excess,service_level,capacity_used,demand_multiplier,capacity_rate,holding_cost_rate,stockout_cost_multiplier,safety_stock_rate
0,Base Case,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,2.0,0.2
1,Peak Demand,Optimal,346.381198,3696.936912,4436.324294,4214.508080,4214.508079,3167,148.066172,1195.574251,0.953247,1.000000,1.2,0.95,0.15,2.0,0.2
2,Capacity Constraint,Optimal,1634.645594,3080.780760,3696.936912,2957.549530,2957.549529,3167,625.525758,416.075287,0.802486,1.000000,1.0,0.80,0.15,2.0,0.2
3,High Stockout Penalty,Optimal,577.301993,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,4.0,0.2
4,High Holding Cost,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.30,2.0,0.2
5,Aggressive Service Strategy,Optimal,0.000000,3080.780760,4005.014988,4405.516487,4005.014993,3167,13.059738,851.074732,0.995876,0.909091,1.0,1.10,0.15,4.0,0.3


In [8]:
# Rank scenarios by service level
scenario_summary.sort_values(
    "service_level",
    ascending=True
)

,scenario,solver_status,objective_value,total_forecast_demand,total_target_inventory,warehouse_capacity,total_recommended_inventory,total_actual_demand,total_actual_shortage,total_actual_excess,service_level,capacity_used,demand_multiplier,capacity_rate,holding_cost_rate,stockout_cost_multiplier,safety_stock_rate
2,Capacity Constraint,Optimal,1634.645594,3080.780760,3696.936912,2957.549530,2957.549529,3167,625.525758,416.075287,0.802486,1.000000,1.0,0.80,0.15,2.0,0.2
0,Base Case,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,2.0,0.2
3,High Stockout Penalty,Optimal,577.301993,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,4.0,0.2
4,High Holding Cost,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.30,2.0,0.2
1,Peak Demand,Optimal,346.381198,3696.936912,4436.324294,4214.508080,4214.508079,3167,148.066172,1195.574251,0.953247,1.000000,1.2,0.95,0.15,2.0,0.2
5,Aggressive Service Strategy,Optimal,0.000000,3080.780760,4005.014988,4405.516487,4005.014993,3167,13.059738,851.074732,0.995876,0.909091,1.0,1.10,0.15,4.0,0.3


In [9]:
# Rank scenarios by total optimization cost
scenario_summary.sort_values(
    "objective_value",
    ascending=False
)

,scenario,solver_status,objective_value,total_forecast_demand,total_target_inventory,warehouse_capacity,total_recommended_inventory,total_actual_demand,total_actual_shortage,total_actual_excess,service_level,capacity_used,demand_multiplier,capacity_rate,holding_cost_rate,stockout_cost_multiplier,safety_stock_rate
2,Capacity Constraint,Optimal,1634.645594,3080.780760,3696.936912,2957.549530,2957.549529,3167,625.525758,416.075287,0.802486,1.000000,1.0,0.80,0.15,2.0,0.2
3,High Stockout Penalty,Optimal,577.301993,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,4.0,0.2
1,Peak Demand,Optimal,346.381198,3696.936912,4436.324294,4214.508080,4214.508079,3167,148.066172,1195.574251,0.953247,1.000000,1.2,0.95,0.15,2.0,0.2
0,Base Case,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.15,2.0,0.2
4,High Holding Cost,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,0.935236,1.000000,1.0,0.95,0.30,2.0,0.2
5,Aggressive Service Strategy,Optimal,0.000000,3080.780760,4005.014988,4405.516487,4005.014993,3167,13.059738,851.074732,0.995876,0.909091,1.0,1.10,0.15,4.0,0.3


In [10]:
# Create a clean summary table for interpretation
scenario_summary_display = scenario_summary.copy()

scenario_summary_display["service_level_pct"] = (
    scenario_summary_display["service_level"] * 100
)

scenario_summary_display["capacity_used_pct"] = (
    scenario_summary_display["capacity_used"] * 100
)

scenario_summary_display = scenario_summary_display[
    [
        "scenario",
        "solver_status",
        "objective_value",
        "total_forecast_demand",
        "total_target_inventory",
        "warehouse_capacity",
        "total_recommended_inventory",
        "total_actual_demand",
        "total_actual_shortage",
        "total_actual_excess",
        "service_level_pct",
        "capacity_used_pct",
    ]
]

scenario_summary_display

,scenario,solver_status,objective_value,total_forecast_demand,total_target_inventory,warehouse_capacity,total_recommended_inventory,total_actual_demand,total_actual_shortage,total_actual_excess,service_level_pct,capacity_used_pct
0,Base Case,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,93.523610,100.000000
1,Peak Demand,Optimal,346.381198,3696.936912,4436.324294,4214.508080,4214.508079,3167,148.066172,1195.574251,95.324718,100.000000
2,Capacity Constraint,Optimal,1634.645594,3080.780760,3696.936912,2957.549530,2957.549529,3167,625.525758,416.075287,80.248634,100.000000
3,High Stockout Penalty,Optimal,577.301993,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,93.523610,100.000000
4,High Holding Cost,Optimal,288.650996,3080.780760,3696.936912,3512.090066,3512.090073,3167,205.107258,550.197331,93.523610,100.000000
5,Aggressive Service Strategy,Optimal,0.000000,3080.780760,4005.014988,4405.516487,4005.014993,3167,13.059738,851.074732,99.587631,90.909091


In [11]:
# Visualize total optimization cost by scenario
fig = px.bar(
    scenario_summary,
    x="scenario",
    y="objective_value",
    title="Total Optimization Cost by Scenario",
    text_auto=".2f",
)

fig.update_layout(
    xaxis_title="Scenario",
    yaxis_title="Objective Value",
    xaxis_tickangle=-30,
)

fig.show()

In [12]:
# Save cost comparison chart
fig.write_html(
    "../outputs/figures/scenario_total_cost.html"
)

In [13]:
# Visualize service level by scenario
service_plot = scenario_summary.copy()

service_plot["service_level_pct"] = (
    service_plot["service_level"] * 100
)

fig = px.bar(
    service_plot,
    x="scenario",
    y="service_level_pct",
    title="Service Level by Scenario",
    text_auto=".2f",
)

fig.update_layout(
    xaxis_title="Scenario",
    yaxis_title="Service Level (%)",
    xaxis_tickangle=-30,
)

fig.show()

In [14]:
# Save service level chart
fig.write_html(
    "../outputs/figures/scenario_service_level.html"
)

In [15]:
# Visualize total actual shortage by scenario
fig = px.bar(
    scenario_summary,
    x="scenario",
    y="total_actual_shortage",
    title="Total Actual Shortage by Scenario",
    text_auto=".2f",
)

fig.update_layout(
    xaxis_title="Scenario",
    yaxis_title="Shortage Units",
    xaxis_tickangle=-30,
)

fig.show()

In [16]:
# Save shortage chart
fig.write_html(
    "../outputs/figures/scenario_shortage.html"
)

In [17]:
# Visualize capacity utilization by scenario
capacity_plot = scenario_summary.copy()

capacity_plot["capacity_used_pct"] = (
    capacity_plot["capacity_used"] * 100
)

fig = px.bar(
    capacity_plot,
    x="scenario",
    y="capacity_used_pct",
    title="Capacity Utilization by Scenario",
    text_auto=".2f",
)

fig.update_layout(
    xaxis_title="Scenario",
    yaxis_title="Capacity Used (%)",
    xaxis_tickangle=-30,
)

fig.show()

In [18]:
# Save capacity utilization chart
fig.write_html(
    "../outputs/figures/scenario_capacity_utilization.html"
)

In [19]:
# Create a scenario risk ranking
scenario_risk_ranking = scenario_summary.copy()

scenario_risk_ranking["service_level_pct"] = (
    scenario_risk_ranking["service_level"] * 100
)

scenario_risk_ranking["capacity_used_pct"] = (
    scenario_risk_ranking["capacity_used"] * 100
)

scenario_risk_ranking = scenario_risk_ranking[
    [
        "scenario",
        "objective_value",
        "total_actual_shortage",
        "total_actual_excess",
        "service_level_pct",
        "capacity_used_pct",
    ]
].sort_values(
    ["total_actual_shortage", "objective_value"],
    ascending=[False, False]
)

scenario_risk_ranking

,scenario,objective_value,total_actual_shortage,total_actual_excess,service_level_pct,capacity_used_pct
2,Capacity Constraint,1634.645594,625.525758,416.075287,80.248634,100.000000
3,High Stockout Penalty,577.301993,205.107258,550.197331,93.523610,100.000000
0,Base Case,288.650996,205.107258,550.197331,93.523610,100.000000
4,High Holding Cost,288.650996,205.107258,550.197331,93.523610,100.000000
1,Peak Demand,346.381198,148.066172,1195.574251,95.324718,100.000000
5,Aggressive Service Strategy,0.000000,13.059738,851.074732,99.587631,90.909091


In [20]:
# Save scenario risk ranking
scenario_risk_ranking.to_csv(
    "../outputs/reports/scenario_risk_ranking.csv",
    index=False
)

In [21]:
# Identify item-store combinations with the highest actual shortages
highest_shortage_items = (
    scenario_detailed_results
    .sort_values(
        "actual_shortage",
        ascending=False
    )
    .head(20)
)

highest_shortage_items[
    [
        "scenario",
        "item_id",
        "store_id",
        "state_id",
        "forecast_demand",
        "actual_demand",
        "recommended_inventory",
        "actual_shortage",
        "actual_excess",
    ]
]

,scenario,item_id,store_id,state_id,forecast_demand,actual_demand,recommended_inventory,actual_shortage,actual_excess
295,Capacity Constraint,HOUSEHOLD_1_327,CA_3,CA,207.284470,250,0.00000,250.00000,0.0
233,Capacity Constraint,FOODS_3_288,CA_1,CA,474.907930,474,381.67883,92.32117,0.0
120,Peak Demand,HOBBIES_1_261,TX_1,TX,56.282690,68,0.00000,68.00000,0.0
420,High Holding Cost,HOBBIES_1_261,TX_1,TX,56.282690,68,0.00000,68.00000,0.0
320,High Stockout Penalty,HOBBIES_1_261,TX_1,TX,56.282690,68,0.00000,68.00000,0.0
220,Capacity Constraint,HOBBIES_1_261,TX_1,TX,56.282690,68,0.00000,68.00000,0.0
20,Base Case,HOBBIES_1_261,TX_1,TX,56.282690,68,0.00000,68.00000,0.0
456,High Holding Cost,HOUSEHOLD_2_342,TX_2,TX,52.785206,60,0.00000,60.00000,0.0
56,Base Case,HOUSEHOLD_2_342,TX_2,TX,52.785206,60,0.00000,60.00000,0.0
356,High Stockout Penalty,HOUSEHOLD_2_342,TX_2,TX,52.785206,60,0.00000,60.00000,0.0


In [22]:
# Save highest shortage item-store combinations
highest_shortage_items.to_csv(
    "../outputs/reports/highest_shortage_items.csv",
    index=False
)

The scenario analysis tested the inventory optimization model under multiple business conditions, including demand surges, capacity constraints, higher stockout penalties, higher holding costs, and aggressive service assumptions.

### Key Findings

- The Base Case establishes the standard inventory allocation strategy under normal demand and capacity assumptions.
- The Peak Demand scenario shows how the inventory plan responds when forecast demand increases.
- The Capacity Constraint scenario tests whether limited inventory capacity creates service-level risk.
- The High Stockout Penalty scenario evaluates how the optimizer responds when service failures become more expensive.
- The High Holding Cost scenario evaluates the trade-off between reducing excess inventory and maintaining service levels.
- The Aggressive Service Strategy tests whether higher safety stock improves availability at the cost of additional inventory investment.